In [4]:
import requests
from pathlib import Path
 
URL = "https://raw.githubusercontent.com/bigdatawirtz/datasets/main/datasets/meteo_alvedro_2021_2025.csv"
resposta = requests.get(URL, timeout=30)
resposta.raise_for_status()
destino = Path("../datos/raw/meteo_alvedro_2021_2025.csv")
destino.write_bytes(resposta.content)   # bytes: tal como chega


81825

In [21]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


RUTA = Path("../datos/raw/meteo_alvedro_2021_2025.csv")

try:
    meteo =pd.read_csv(
    RUTA,
    sep=";",
    skiprows=6,
    decimal=",",
    thousands=".",
)
except Exception as e:
    print(type(e).__name__, "→", e)
    
# Imprimir a informacion do dataframe
meteo.info()

# Renomear columnas
meteo = meteo.rename(columns={
    "Data": "data",
    "Temp. media (�C)": "tmed",
    "Temp. m�xima (�C)": "tmax",
    "Temp. m�nima (�C)": "tmin",
    "Precipitaci�n (mm)": "prec",
    "Vento medio (km/h)": "vento",
    "Humidade relativa (%)": "hum",
    "Presi�n (hPa)": "pres",
})
meteo.columns.tolist()

meteo.dtypes



<class 'pandas.DataFrame'>
RangeIndex: 1812 entries, 0 to 1811
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Data                   1812 non-null   str    
 1   Temp. media (�C)       1803 non-null   float64
 2   Temp. m�nima (�C)      1806 non-null   float64
 3   Temp. m�xima (�C)      1803 non-null   float64
 4   Humidade relativa (%)  1812 non-null   int64  
 5   Precipitaci�n (mm)     1804 non-null   str    
 6   Vento medio (km/h)     1812 non-null   float64
 7   Presi�n (hPa)          1811 non-null   float64
dtypes: float64(5), int64(1), str(2)
memory usage: 113.4 KB


data         str
tmed     float64
tmin     float64
tmax     float64
hum        int64
prec         str
vento    float64
pres     float64
dtype: object

In [22]:
# Renomear columnas
meteo = meteo.rename(columns={
    "Data": "data",
    "Temp. media (�C)": "tmed",
    "Temp. m�xima (�C)": "tmax",
    "Temp. m�nima (�C)": "tmin",
    "Precipitaci�n (mm)": "prec",
    "Vento medio (km/h)": "vento",
    "Humidade relativa (%)": "hum",
    "Presi�n (hPa)": "pres",
})
meteo.columns.tolist()

['data', 'tmed', 'tmin', 'tmax', 'hum', 'prec', 'vento', 'pres']

In [24]:

# Pasar data a formato datetime
meteo["data"] = pd.to_datetime(meteo["data"], format="mixed", dayfirst=True, errors="coerce")
# Pasar precipitacion a formato numerico
meteo["prec"] = pd.to_numeric(
    meteo["prec"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)
meteo.loc[meteo["prec"] < 0, "prec"] = pd.NA

# Comprobar filas duplicadas
print("Filas duplicadas:", meteo.duplicated().sum())
meteo = meteo.drop_duplicates()


meteo.describe().round(1)

Filas duplicadas: 0


,data,tmed,tmin,tmax,hum,prec,vento,pres
count,1804,1795.0,1798.0,1795.0,1804.0,1674.0,1804.0,1803.0
mean,2023-07-06 15:46:41.773836,-2.2,-23.1,2.5,80.2,3.1,11.9,1018.2
min,2021-01-01 00:00:00,-9999.0,-9999.0,-9999.0,36.0,0.0,3.2,987.9
25%,2022-03-28 18:00:00,11.3,7.1,15.4,76.0,0.0,8.1,1014.4
50%,2023-08-07 12:00:00,14.7,10.9,18.9,81.0,0.2,10.4,1018.5
75%,2024-10-07 06:00:00,18.0,14.0,22.8,86.0,3.2,14.9,1022.5
max,2025-12-31 00:00:00,31.3,21.1,39.2,100.0,58.7,38.5,1037.9
std,NaN,409.2,577.4,409.4,8.1,6.2,5.2,7.4


In [27]:
temperaturas = ["tmed", "tmin", "tmax"]
imposibles = meteo[temperaturas].eq(-9999)
for columna in temperaturas:
    imposibles[columna] |= meteo[columna] > 50
print("Valores imposibles:", imposibles.sum())
meteo[temperaturas] = meteo[temperaturas].mask(imposibles)
meteo[temperaturas] = meteo[temperaturas].interpolate(method="linear")

print(meteo["prec"])

Valores imposibles: tmed    0
tmin    0
tmax    0
dtype: int64
0       12.2
1        9.8
2       12.4
3        4.7
4        1.2
        ... 
1799     1.3
1800     3.6
1806     0.0
1808     0.0
1809     2.9
Name: prec, Length: 1804, dtype: Float64
